<a href="https://colab.research.google.com/github/Maja-GHub/Ai-Agent-Companion/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.5-flash-lite', 'google/gemini-3.5-flash']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [2]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.


In [5]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

InternalServerError: Error code: 503 - {'message': 'The requested model is currently unavailable.', 'type': 'server_error'}

For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [7]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

Elara lived in a house that always felt a little too big, a little too quiet. Her parents were kind, but busy, and since moving to the new town, Elara hadn’t quite found her place. She spent most afternoons curled up with a book, her imagination her only friend.

One sweltering summer day, escaping the heat, she ventured into the overgrown garden behind their house. Hidden behind a tangle of rose bushes and forgotten weeds, she discovered it: an ancient stone well, half-covered by a thick slab of mossy wood. Beside it, reaching towards the sky like a benevolent giant, stood an oak tree older than the town itself.

Its bark was gnarled and silvered, its leaves a vibrant, whispering green. As Elara approached, a gentle breeze stirred the canopy, and she distinctly heard a soft, almost imperceptible murmur. Not words, but a sound that felt like a question, an invitation.

Cautiously, she sat at the base of its enormous trunk, her back pressed against the cool, rough bark. The air around t

In [14]:
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def _should_not_have_leading_space(self, word):
        """
        Checks if a word should not have a leading space when following another word.
        This includes punctuation and contractions typically attached to the preceding word.
        """
        if not word:
            return False

        # Common punctuation marks that usually don't have a leading space
        punctuation_marks = {",", ".", ";", ":", "!", "?", ")", "]", "}"}
        if word[0] in punctuation_marks:
            return True

        # Common contractions/possessives that should stick to the preceding word
        # Assumes they are tokenized as separate "words" like "'s" or "n't"
        if word.lower() in {"'s", "'m", "'d", "'ll", "'ve", "'re", "n't"}:
            return True

        # Multi-character punctuation
        if word == "..." or word == "…":
            return True

        return False

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            # Find end of word (token separated by space or newline)
            while i < n and text_chunk[i] not in ' \n':
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                # Determine if a space should precede the current_word.
                # A space is generally needed if it's not the first word on the line
                # AND the current word is not a punctuation/contraction token
                # that should directly follow the previous word without a space.
                add_leading_space = False
                if self.current_line_length > 0: # Not the very beginning of a line
                    if not self._should_not_have_leading_space(current_word):
                        add_leading_space = True

                # Calculate length with potential leading space
                potential_len = self.current_line_length + (1 if add_leading_space else 0) + len(current_word)

                # Case 1: Word itself is too long for a line (force hyphenation)
                if len(current_word) > self.max_length:
                    if self.current_line_length > 0: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0

                    # Break the long word with hyphens
                    remaining_word = current_word
                    while len(remaining_word) > self.max_length:
                        part_to_print = remaining_word[:self.max_length - 1]
                        sys.stdout.write(part_to_print + '-')
                        sys.stdout.write('\n')
                        self.current_line_length = 0 # Reset for new line
                        remaining_word = remaining_word[self.max_length - 1:]

                    # Print the last part of the word
                    sys.stdout.write(remaining_word)
                    self.current_line_length += len(remaining_word)

                # Case 2: Word doesn't fit on current line (print on new line)
                elif potential_len > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if add_leading_space:
                        sys.stdout.write(' ')
                        self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If current_line_length is exactly max_length, and we received a space delimiter,
                # it means the current word filled the line, and the next word should go on a new line.
                # So we implicitly wrap here.
                if self.current_line_length == self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()

In [8]:
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.5-flash-lite', stream=True):
  wrapper.print(chunk)

The tapestry of Roman history is not merely a chronological account of conquests
and emperors; it is a sprawling, intricate narrative of transformation,
adaptation, and the relentless march of human ambition. To truly grasp the
evolution of the Roman Empire is to embark on a journey that spans over a
thousand years, witnessing a humble village on the Tiber River blossom into a
Mediterranean hegemon, then fracture and ultimately fade, leaving an indelible
mark on the very fabric of Western civilization.

Our saga begins not with legions and senators, but with the whisper of myths and
the glint of bronze. The earliest Romans, a Latin people inhabiting the Latium
region of Italy, were farmers and shepherds, carving out a precarious existence
in a land dotted with competing tribal settlements. This period, the **Regal
Period (c. 753 – 509 BCE)**, is shrouded in legend, dominated by seven kings,
the most famous being Romulus, the city's mythical founder. While historical
accuracy is debated

In [9]:
wrapper = LineWrapper()
for chunk in ai.generate_text('Summarize the evolution of the Roman Empire.', model_name='google/gemini-2.5-flash-lite', stream=True):
  wrapper.print(chunk)

The Roman Empire's evolution is a monumental saga spanning over a thousand
years, marked by a dramatic transformation from a modest city-state to a vast,
complex civilization that profoundly shaped Western history. Here's a summary of
its key stages:

**1. The Roman Republic (509 BCE - 27 BCE): The Foundation of Power and
Expansion**

* **Origins :** Rome began as a small city-state in Italy, traditionally founded
in 753 BCE. After overthrowing its monarchy, it established a republic governed
by elected officials (consuls) and a powerful Senate.
* **Internal Development:** The Republic was characterized by a constant
struggle between the patrician (aristocratic) and plebeian (commoner) classes,
leading to significant legal and social reforms like the Twelve Tables.
* **Military Prowess and Expansion:** Driven by ambition and a highly
disciplined military, Rome gradually conquered the Italian peninsula. This
expansion brought them into conflict with major powers like Carthage (Punic
War

In [10]:
wrapper = LineWrapper()
for chunk in ai.generate_text('Summarize the key factors that led to the decline of the Western Roman Empire.', model_name='google/gemini-2.5-flash-lite', stream=True):
  wrapper.print(chunk)

The decline of the Western Roman Empire was a complex and multifaceted process,
not attributable to a single cause. However, several key factors, often
interconnected, played a significant role in its eventual collapse in 476 CE.
Here's a summary of the most important ones:

**Internal Weaknesses:**

* **Political Instability and Corruption:**
* ** Frequent Civil Wars and Usurpations:** The empire suffered from constant
power struggles and civil wars, weakening its armies and diverting resources.
Emperors were often assassinated or overthrown, leading to a rapid succession of
rulers and a lack of consistent leadership.
* **Ineffective and Corrupt Administration:** As the empire grew, its
bureaucracy became unwieldy and prone to corruption. Officials often prioritized
personal gain over the welfare of the state, leading to inefficiency and a
decline in public services.
* **Lack of Clear Succession:** There was no established system for imperial
succession, which fueled ambition and conf

### Comparison with `textwrap`

Let's compare the output of our custom `LineWrapper` with Python's built-in `textwrap` module. We will use the same text to see the differences, especially concerning hyphenation.

In [12]:
import textwrap

# Generate some text to compare
comparison_text_stream = ai.generate_text(
    'Write a very long and complex sentence that describes the process of photosynthesis, emphasizing the intricate biochemical reactions and the role of various cellular organelles, and make sure it has some exceptionally long words to test the hyphenation.',
    model_name='google/gemini-2.5-flash-lite',
    stream=True
)

# Collect the full text from the stream
full_comparison_text = ""
for chunk in comparison_text_stream:
    full_comparison_text += chunk

print("--- Output from custom LineWrapper (max_length=80) ---")
wrapper = LineWrapper(max_length=80)
wrapper.print(full_comparison_text)
print("\n\n--- Output from textwrap (width=80) ---")
# textwrap.fill automatically handles breaking long words with hyphens by default.
wrapped_text_textwrap = textwrap.fill(full_comparison_text, width=80, drop_whitespace=False, break_long_words=True)
print(wrapped_text_textwrap)

--- Output from custom LineWrapper (max_length=80) ---
Within the chlorophyll-containing chloroplasts, a marvelously orchestrated
biophysical and biochemical cascade, known as photosynthesis, commences with the
light-dependent reactions in the thylakoid membranes where photosystems I and
II, through a series of exquisitely sensitive electron-transport chain
components like plastoquinone, cytochrome b6f complex, and plastocyanin, harness
photons from electromagnetic radiation, initiating photolysis of water molecules
to release oxygen as a gratuitous byproduct and generating high-energy electrons
that, subsequently energized by more photon absorption, are funnelled through a
proton gradient across the thylakoid lumen, ultimately driving ATP synthase to
produce adenosine triphosphate, while simultaneously reducing NADP+ to NADPH,
which then, along with the aforementioned ATP, serve as the essential energy
currency and reducing power for the light-independent Calvin cycle occurring in
the

### Test Case: Short Words with `LineWrapper`

Let's test the `LineWrapper` with a sentence containing only short words to ensure it handles them correctly without unnecessary hyphenation, just wrapping them by spaces.

In [13]:
# Generate a sentence with short words
short_word_text_stream = ai.generate_text(
    'The quick brown fox jumps over the lazy dog. This is a simple test sentence with short words.',
    model_name='google/gemini-2.5-flash-lite',
    stream=True
)

# Collect the full text from the stream
full_short_word_text = ""
for chunk in short_word_text_stream:
    full_short_word_text += chunk

print("--- Output from custom LineWrapper (max_length=20) for short words ---")
# Using a smaller max_length to ensure wrapping occurs even with short words
wrapper_short = LineWrapper(max_length=20)
wrapper_short.print(full_short_word_text)

print("\n\n--- Output from textwrap (width=20) for short words ---")
wrapped_text_short_textwrap = textwrap.fill(full_short_word_text, width=20, drop_whitespace=False, break_long_words=False)
print(wrapped_text_short_textwrap)

--- Output from custom LineWrapper (max_length=20) for short words ---
You're absolutely
right!

"The quick brown fox
jumps over the lazy
dog" is a very
famous **pangram**.
A pangram is a
sentence that
contains every
letter of the
alphabet at least
once. It's often
used for testing
typefaces,
keyboards, and other
text-related
applications because
it ensures all
characters are
displayed and
functional.

And yes, it's also a
good example of a
sentence with
relatively **short
and common words**,
making it easy to
understand and
pronounce.

--- Output from textwrap (width=20) for short words ---
You're absolutely 
right!  "The quick 
brown fox jumps over
 the lazy dog" is a 
very famous 
**pangram**. A 
pangram is a 
sentence that 
contains every 
letter of the 
alphabet at least 
once. It's often 
used for testing 
typefaces, 
keyboards, and other
 text-related 
applications because
 it ensures all 
characters are 
displayed and 
functional.  And 
yes, it's also a 
good example of a 
sent

### Re-running Comparison with `textwrap` (after `LineWrapper` refactor)

Now that `LineWrapper` has been refactored to handle punctuation more intelligently, let's re-run the long text comparison to observe the changes, especially in how punctuation interacts with line breaks.

In [15]:
import textwrap

# Generate some text to compare (same as before)
comparison_text_stream = ai.generate_text(
    'Write a very long and complex sentence that describes the process of photosynthesis, emphasizing the intricate biochemical reactions and the role of various cellular organelles, and make sure it has some exceptionally long words to test the hyphenation.',
    model_name='google/gemini-2.5-flash-lite',
    stream=True
)

# Collect the full text from the stream
full_comparison_text = ""
for chunk in comparison_text_stream:
    full_comparison_text += chunk

print("--- Output from custom LineWrapper (max_length=80) - AFTER REFACTOR ---")
wrapper = LineWrapper(max_length=80)
wrapper.print(full_comparison_text)
print("\n\n--- Output from textwrap (width=80) ---")
# textwrap.fill automatically handles breaking long words with hyphens by default.
wrapped_text_textwrap = textwrap.fill(full_comparison_text, width=80, drop_whitespace=False, break_long_words=True)
print(wrapped_text_textwrap)

--- Output from custom LineWrapper (max_length=80) - AFTER REFACTOR ---
Harnessing the sun's life-sustaining electromagnetic radiation, plant cells,
specifically within their chloroplasts, embark upon the astonishingly complex,
photo-biochemical ballet of photosynthesis, whereby light-dependent reactions,
occurring across the thylakoid membranes where chlorophyll molecules absorb
photons, initiate an intricate cascade of electron transport, proton pumping,
and the photolysis of water, ultimately generating adenosine triphosphate (ATP)
and nicotinamide adenine dinucleotide phosphate (NADPH)—energy-rich molecules
that then fuel the subsequent, light-independent Calvin cycle, or C3 cycle,
within the stroma, a process characterized by a series of enzyme-catalyzed
transformations, including carbon fixation via the Rubisco enzyme's
carboxylation of ribulose-1,5-bisphosphate, followed by reduction and
regeneration steps, culminating in the synthesis of glucose, a fundamental
carbohydrate, fro

### Comparison with `textwrap`: Handling URLs

Let's test both `LineWrapper` and `textwrap` with a paragraph that includes URLs. URLs are typically treated as single, unbreakable words, which can be a challenge for text wrapping algorithms, especially concerning line breaks and how they might affect the URL's integrity.

In [ ]:
import textwrap

# Generate a paragraph with URLs
url_text_stream = ai.generate_text(
    "Here is an example of a very long sentence with some important information and a URL: For more details on advanced AI applications, please visit our comprehensive resource page at https://www.example-very-long-domain-name.com/some/path/to/an/exceptionally/detailed/article/about/artificial-intelligence-and-machine-learning-algorithms. You can also check out our blog for updates at http://blog.example.com. This demonstrates how URL handling can impact text formatting.",
    model_name='google/gemini-2.5-flash-lite',
    stream=True
)

# Collect the full text from the stream
full_url_text = ""
for chunk in url_text_stream:
    full_url_text += chunk

print("--- Output from custom LineWrapper (max_length=80) for URLs ---")
wrapper_url = LineWrapper(max_length=80)
wrapper_url.print(full_url_text)

print("\n\n--- Output from textwrap (width=80, break_long_words=False) for URLs ---")
# For textwrap, we explicitly set break_long_words=False to see how it handles URLs without internal breaks
wrapped_text_url_textwrap = textwrap.fill(full_url_text, width=80, drop_whitespace=False, break_long_words=False)
print(wrapped_text_url_textwrap)

print("\n\n--- Output from textwrap (width=80, break_long_words=True) for URLs ---")
# With break_long_words=True, textwrap will break long words/URLs if necessary
wrapped_text_url_textwrap_break = textwrap.fill(full_url_text, width=80, drop_whitespace=False, break_long_words=True)
print(wrapped_text_url_textwrap_break)